# Weeks 1–2 · Solutions

Reference implementations. Each check cell is identical to the exercise notebook.

In [ ]:
import csv, time
from pathlib import Path
from functools import wraps
DATA = next(p / "data" / "generated" for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "make_data.py").exists())

### Exercise 1 · Comprehensions

In [ ]:
def honours_pct(pairs):
    return {name: round(g / 4 * 100, 1) for name, g in pairs if g >= 3.0}

In [ ]:
assert honours_pct([("a", 3.0), ("b", 2.9), ("c", 4.0)]) == {"a": 75.0, "c": 100.0}
assert honours_pct([]) == {}
print("✓ exercise 1")

### Exercise 2 · Flatten

In [ ]:
def flatten2(nested):
    return [x for level1 in nested for level2 in level1 for x in level2]

In [ ]:
assert flatten2([[[1], [2, 3]], [[4]]]) == [1, 2, 3, 4]
assert flatten2([[[]]]) == []
print("✓ exercise 2")

### Exercise 3 · Generator over a file

In [ ]:
def warnings(path):
    with open(path) as f:
        for line in f:
            if " WARNING " in line:
                yield int(line.split("user=")[1].split()[0])

In [ ]:
import types
g = warnings(DATA / "server.log")
assert isinstance(g, types.GeneratorType)
ids = list(g)
assert all(isinstance(i, int) for i in ids) and 50 < len(ids) < 200, len(ids)
print("✓ exercise 3 -", len(ids), "warnings")

### Exercise 4 · Lazy running mean

In [ ]:
def running_mean(values):
    total, n = 0.0, 0
    for v in values:
        total += v
        n += 1
        yield total / n

In [ ]:
from itertools import count, islice
assert list(running_mean([2, 4, 6])) == [2.0, 3.0, 4.0]
assert list(islice(running_mean(count(1)), 4)) == [1.0, 1.5, 2.0, 2.5]   # infinite input
print("✓ exercise 4")

### Exercise 5 · Sorting with keys

In [ ]:
def rank(rows):
    return sorted(rows, key=lambda r: (r[1], -r[2], r[0]))

In [ ]:
rows = [("Omar", "CS", 3.1), ("Sara", "DS", 3.9), ("Ahmad", "CS", 3.1), ("Lina", "CS", 3.8)]
assert rank(rows) == [("Lina", "CS", 3.8), ("Ahmad", "CS", 3.1), ("Omar", "CS", 3.1), ("Sara", "DS", 3.9)]
print("✓ exercise 5")

### Exercise 6 · A counting decorator

In [ ]:
def count_calls(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        return func(*args, **kwargs)
    wrapper.calls = 0
    return wrapper

In [ ]:
@count_calls
def square(x):
    """doc"""
    return x * x
assert [square(2), square(3)] == [4, 9]
assert square.calls == 2 and square.__name__ == "square" and square.__doc__ == "doc"
print("✓ exercise 6")

### Exercise 7 · Safe parsing

In [ ]:
def to_float(s, default=None):
    if s is None:
        return default
    try:
        return float(str(s).replace(",", "."))
    except ValueError:
        return default

In [ ]:
assert to_float("3.5") == 3.5 and to_float("3,1") == 3.1
assert to_float("N/A") is None and to_float("", default=0.0) == 0.0 and to_float(None, -1) == -1
print("✓ exercise 7")

### Exercise 8 · Typed record + validation

In [ ]:
from dataclasses import dataclass

@dataclass
class Course:
    code: str
    title: str
    credits: int

    def __post_init__(self):
        if not (len(self.code) == 6 and self.code.isdigit()):
            raise ValueError(f"bad course code: {self.code!r}")
        if not 1 <= self.credits <= 6:
            raise ValueError(f"credits out of range: {self.credits}")

In [ ]:
c = Course("606315", "Programming for DS", 3)
assert (c.code, c.credits) == ("606315", 3)
for bad in [("60631", "x", 3), ("606315", "x", 0), ("606315", "x", 7)]:
    try:
        Course(*bad); raise AssertionError(f"should have failed: {bad}")
    except ValueError:
        pass
print("✓ exercise 8")

### Exercise 9 · Context manager

In [ ]:
import os
from contextlib import contextmanager

@contextmanager
def cd(path):
    previous = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)

In [ ]:
import os
start = os.getcwd()
with cd(".."):
    assert os.getcwd() != start
assert os.getcwd() == start
try:
    with cd(".."):
        raise RuntimeError("boom")
except RuntimeError:
    pass
assert os.getcwd() == start
print("✓ exercise 9")

### Exercise 10 · Mini ETL

In [ ]:
def gpa_by_major(path):
    acc = {}
    with open(path, newline="") as f:
        for row in csv.DictReader(f):
            g = to_float(row["gpa"])
            if g is None:
                continue
            acc.setdefault(row["major"].strip(), []).append(g)
    return {m: (len(v), round(sum(v) / len(v), 2)) for m, v in acc.items()}

In [ ]:
res = gpa_by_major(DATA / "students.csv")
assert set(res) == {"Data Science", "Computer Science", "Cybersecurity", "AI", "Software Eng"}, set(res)
assert all(isinstance(v, tuple) and len(v) == 2 for v in res.values())
assert sum(c for c, _ in res.values()) > 180
assert all(1.5 <= m <= 4.0 for _, m in res.values())
print("✓ exercise 10")
for k, (n, m) in sorted(res.items()): print(f"  {k:18s} n={n:3d} mean={m}")